# Developing Simple Supervised and Unsupervised Learning Models

## 📚 Learning Objectives

By completing this notebook, you will:
- Develop simple supervised learning models
- Develop simple unsupervised learning models
- Use scikit-learn for ML
- Train and evaluate models
- Apply all three model families to **real, published datasets** that ship with scikit-learn

## 🔗 Where this fits

**Builds on:** Unit 2, lesson 05 "Introduction to Machine Learning" — the paradigms named there are trained here for the first time.

**Used later in:** Course 04 (AIAT 114) — Units 1, 3 and 4, which take regression, classification and clustering apart in detail.

---

This notebook covers practical activities from **Course 01, Unit 2**:
- Developing simple supervised and unsupervised learning models

---

## Introduction

**Supervised learning** uses labeled data to train models, while **unsupervised learning** finds patterns in unlabeled data. Both are fundamental to machine learning.


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# Setup: load three REAL datasets, one per task — regression (numeric target),
# classification (binary target), clustering (no target at all) — so we can compare the
# model families fairly. All three ship inside scikit-learn, so they work offline.
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes, load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, accuracy_score

print("✅ Libraries imported!")
print("\nSupervised and Unsupervised Learning Models")
print("=" * 60)

# REGRESSION DATA — real: 442 diabetes patients, 10 baseline measurements
# (age, sex, BMI, blood pressure, six blood serum values). The target is a real
# quantitative measure of disease progression one year after baseline.
diabetes = load_diabetes()
X_regression, y_regression = diabetes.data, diabetes.target

# CLASSIFICATION DATA — real: 569 breast-tumour biopsies from the Wisconsin
# Diagnostic study, 30 measurements taken from digitised cell images.
# Target: 0 = malignant, 1 = benign — an actual clinical decision.
cancer = load_breast_cancer()
X_classification, y_classification = cancer.data, cancer.target

# CLUSTERING DATA — real: 150 iris flowers measured by Edgar Anderson (1935).
# We deliberately THROW AWAY the species labels: unsupervised learning never sees them.
# We keep them in `iris_true_species` only so we can check afterwards how well the
# clusters happened to line up with reality.
iris = load_iris()
X_clustering = iris.data
iris_true_species = iris.target

print("\n✅ Real datasets loaded!")
print(f"  Regression     : diabetes  {X_regression.shape[0]} patients x {X_regression.shape[1]} features")
print(f"                   target range {y_regression.min():.0f} to {y_regression.max():.0f} (disease progression)")
print(f"  Classification : breast cancer  {X_classification.shape[0]} biopsies x {X_classification.shape[1]} features")
print(f"                   class balance: {(y_classification == 0).sum()} malignant / {(y_classification == 1).sum()} benign")
print(f"  Clustering     : iris  {X_clustering.shape[0]} flowers x {X_clustering.shape[1]} features (labels hidden)")

✅ Libraries imported!

Supervised and Unsupervised Learning Models

✅ Real datasets loaded!
  Regression     : diabetes  442 patients x 10 features
                   target range 25 to 346 (disease progression)
  Classification : breast cancer  569 biopsies x 30 features
                   class balance: 212 malignant / 357 benign
  Clustering     : iris  150 flowers x 4 features (labels hidden)


### 🧰 Bridge: why split data into train and test sets?

`train_test_split` below holds out 20% of the data that the model **never sees during
training**. We then score the model only on that held-out 20%: a good score there means
the model learned a pattern that **generalizes** to new data, instead of memorizing the
examples it was shown. Unit 4 (notebooks 04 and 07) shows what goes wrong when a model
memorizes — overfitting — and how to detect it with these held-out scores.


In [2]:
# Supervised Learning: Regression
print("=" * 60)
print("SUPERVISED LEARNING: REGRESSION")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X_regression, y_regression, test_size=0.2, random_state=42
)

# Train linear regression
reg_model = LinearRegression()
reg_model.fit(X_train, y_train)

# Predictions
y_pred = reg_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

print(f"\nRegression Model (real diabetes data):")
print(f"  MSE: {mse:.4f}")
print(f"  R² Score: {reg_model.score(X_test, y_test):.4f}")
print(f"  Baseline MSE if we always predicted the mean: {((y_test - y_train.mean()) ** 2).mean():.4f}")

print("\n✅ Regression model trained!")
print("\n⚠️ Read that R² honestly. On real biomedical data a linear model explains")
print("   only part of the variance — disease progression depends on much more than")
print("   ten baseline measurements. A textbook R² near 1.0 is a sign of invented data,")
print("   not of a good model.")

SUPERVISED LEARNING: REGRESSION

Regression Model (real diabetes data):
  MSE: 2900.1936
  R² Score: 0.4526
  Baseline MSE if we always predicted the mean: 5361.5335

✅ Regression model trained!

⚠️ Read that R² honestly. On real biomedical data a linear model explains
   only part of the variance — disease progression depends on much more than
   ten baseline measurements. A textbook R² near 1.0 is a sign of invented data,
   not of a good model.


In [3]:
# Supervised Learning: Classification
print("=" * 60)
print("SUPERVISED LEARNING: CLASSIFICATION")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X_classification, y_classification, test_size=0.2, random_state=42,
    stratify=y_classification   # keep the real malignant/benign ratio in both halves
)

# Real features live on wildly different scales (cell area in the hundreds,
# smoothness around 0.1), so we standardise before fitting a linear model.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # learn the scaling from TRAIN only
X_test = scaler.transform(X_test)         # apply the same scaling to TEST

# Train logistic regression
clf_model = LogisticRegression(random_state=42, max_iter=5000)
clf_model.fit(X_train, y_train)

# Predictions
y_pred = clf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nClassification Model (real breast-cancer biopsies):")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Majority-class baseline: {max((y_test == 0).mean(), (y_test == 1).mean()):.4f}")
print(f"  (always guessing the more common class would already score this much)")

print("\n✅ Classification model trained!")

SUPERVISED LEARNING: CLASSIFICATION

Classification Model (real breast-cancer biopsies):
  Accuracy: 0.9825
  Majority-class baseline: 0.6316
  (always guessing the more common class would already score this much)

✅ Classification model trained!


In [4]:
# Unsupervised learning: K-Means invents 3 groups from unlabeled points; inertia
# measures how tightly each cluster hugs its center (lower = tighter).
# Unsupervised Learning: Clustering
print("=" * 60)
print("UNSUPERVISED LEARNING: CLUSTERING")
print("=" * 60)

# K-Means clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_clustering)

print(f"\nClustering Model (real iris measurements, species labels hidden):")
print(f"  Number of clusters: 3")
print(f"  Cluster centers (sepal len, sepal wid, petal len, petal wid, in cm):")
for k, centre in enumerate(kmeans.cluster_centers_):
    print(f"    cluster {k}: " + "  ".join(f"{v:5.2f}" for v in centre))
print(f"  Inertia: {kmeans.inertia_:.4f}")

# Now the payoff of using REAL data: we can reveal the hidden species labels and
# see whether the unsupervised clusters actually rediscovered them.
print("\n✅ Clustering model trained!")
print("\n🔎 How well did the clusters match the true species?")
crosstab = pd.crosstab(clusters, pd.Series(iris_true_species).map(dict(enumerate(iris.target_names))),
                       rownames=["cluster"], colnames=["true species"])
print(crosstab)
print("\n💡 K-Means never saw a single label, yet the clusters line up closely with the")
print("   real species — that is unsupervised learning finding genuine structure.")
print("   The overlap that remains is real too: two of the species genuinely overlap in")
print("   petal and sepal size, and no algorithm can separate what nature did not.")

UNSUPERVISED LEARNING: CLUSTERING

Clustering Model (real iris measurements, species labels hidden):
  Number of clusters: 3
  Cluster centers (sepal len, sepal wid, petal len, petal wid, in cm):
    cluster 0:  5.90   2.75   4.39   1.43
    cluster 1:  5.01   3.43   1.46   0.25
    cluster 2:  6.85   3.07   5.74   2.07
  Inertia: 78.8514

✅ Clustering model trained!

🔎 How well did the clusters match the true species?
true species  setosa  versicolor  virginica
cluster                                    
0                  0          48         14
1                 50           0          0
2                  0           2         36

💡 K-Means never saw a single label, yet the clusters line up closely with the
   real species — that is unsupervised learning finding genuine structure.
   The overlap that remains is real too: two of the species genuinely overlap in
   petal and sepal size, and no algorithm can separate what nature did not.


## Summary

This notebook covered:
- ✅ **Supervised Learning**: Regression and classification with labeled data
- ✅ **Unsupervised Learning**: Clustering to find patterns in unlabeled data
- ✅ **Model Training**: Using scikit-learn to train and evaluate models

- ✅ **Real data from the start**: diabetes progression (442 patients), breast-cancer
  biopsies (569 samples) and Anderson's iris measurements (150 flowers)

Both supervised and unsupervised learning are fundamental to machine learning applications.

**The honest takeaway:** a linear model on real biomedical data explains only part of the
variance, real classification has a majority-class baseline you must beat, and real clusters
overlap because nature overlaps. Invented data hides all three of these lessons.

## 📚 References

1. MacQueen, J. (1967). *Some Methods for Classification and Analysis of Multivariate Observations* (k-means). Proceedings of the 5th Berkeley Symposium on Mathematical Statistics and Probability.
2. Cox, D. R. (1958). *The Regression Analysis of Binary Sequences*. Journal of the Royal Statistical Society, Series B, 20(2), 215–242.
3. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.), Chs. 3–4 and 12. Springer.